In [1]:
using LinearAlgebra
using Random
using Printf

In [5]:
Random.seed!(2)
N = 100
n = 5
A = rand(N,N)
# A += A'
A += diagm(1:N)*.2
App = A[1:n, 1:n]
Apq = A[1:n, n+1:N]
Aqp = A[n+1:N, 1:n]
Aqq = A[n+1:N, n+1:N];


In [6]:
e = eigvals(A)
ep = eigvals(App)

println(" Exact spectrum:")
for i in e
    @printf(" %12.8f %12.8fi\n", real(i), imag(i))
end

println(" Pspace spectrum:")
for i in ep
    @printf(" %12.8f %12.8fi\n", real(i), imag(i))
end

 Exact spectrum:
  -2.90852113   0.00000000i
  -2.39834483  -0.07684962i
  -2.39834483   0.07684962i
  -2.28264747  -0.83311792i
  -2.28264747   0.83311792i
  -1.98020212   0.00000000i
  -1.89663862  -0.99627979i
  -1.89663862   0.99627979i
  -1.87196329  -0.54723991i
  -1.87196329   0.54723991i
  -1.86836524  -1.22821598i
  -1.86836524   1.22821598i
  -1.78409308  -1.65506290i
  -1.78409308   1.65506290i
  -1.72284924  -0.53637177i
  -1.72284924   0.53637177i
  -1.56751567  -1.86408442i
  -1.56751567   1.86408442i
  -1.45403158  -2.23668521i
  -1.45403158   2.23668521i
  -1.40127468  -1.27036422i
  -1.40127468   1.27036422i
  -1.38371041   0.00000000i
  -1.23810177  -1.53392872i
  -1.23810177   1.53392872i
  -1.13655930  -0.17956551i
  -1.13655930   0.17956551i
  -0.95888236  -2.27704860i
  -0.95888236   2.27704860i
  -0.79634686  -2.08898025i
  -0.79634686   2.08898025i
  -0.66624315  -1.17433881i
  -0.66624315   1.17433881i
  -0.47778313  -1.62720737i
  -0.47778313   1.62720737i
  -

In [7]:
function Aeff(E)
    return App + Apq * inv(E*I - Aqq) * Aqp
end

Aeff (generic function with 1 method)

# Method 1
We can iterate by choosing the smallest real eigenvalue from each P-space spectrum

In [8]:
ei = eigvals(App)

for i in 1:20
    idx = argmin(real(ei))
    ei = eigvals(Aeff(ei[idx]))
    idx = argmin(real(ei)) 
    @printf(" Iter: %2i %12.8f %12.8fi\n", i, real(ei[idx]), imag(ei[idx]))
end

 Iter:  1  -4.31619168   8.76908640i
 Iter:  2  -0.33755626   0.02297072i
 Iter:  3  -0.33929791   0.26041226i
 Iter:  4  -2.30138563  -2.81958677i
 Iter:  5  -0.24836741   0.37984887i
 Iter:  6  -2.19230492  -1.21534413i
 Iter:  7  -1.07365936  -0.40128921i
 Iter:  8  -2.69907411   0.52271282i
 Iter:  9  -1.01194205  -0.45744254i
 Iter: 10  -3.26478095   0.68597563i
 Iter: 11  -0.54447692  -0.18698847i
 Iter: 12  -5.23694995   5.21686884i
 Iter: 13  -0.29952676   0.00255955i
 Iter: 14  -1.69426512   7.32607176i
 Iter: 15  -0.35431026   0.08010220i
 Iter: 16  -0.50849086   0.31554205i
 Iter: 17  -5.28498885  -4.08896506i
 Iter: 18  -0.29967530   0.01684812i
 Iter: 19  -1.88501231   6.44800321i
 Iter: 20  -0.32624503   0.10152273i


# Method 2
Alternatively, we can iterate by choosing the eigenvector that has maximal overlap with previous iteration

In [254]:
ep, vp = eigen(App)
idx = argmin(real(ep))
vp = vp[:,idx]

for i in 1:20

    vold = vp
    Ai = Aeff(ep[idx])
    ep,vp = eigen(Ai)
    ovlps = real(inv(vp)*vold)
    # ovlps = real(inv(vp)*vold)
    idx = argmax(abs.(ovlps))
    vp = vp[:,idx]
    ovlp = ovlps[idx] 
    # println(ovlps)
    # @printf("iter: %2i idx %i ovlp %12.8f\n", i, idx, ovlp)
    @printf(" Iter: %2i ovlp: %12.8f eig: %12.8f %12.8fi\n", i, ovlp, real(ep[idx]), imag(ep[idx]))
end

 Iter:  1 ovlp:  -1.08082652 eig:  -0.01419949   0.00000000i
 Iter:  2 ovlp:   0.99410919 eig:   0.07723467   0.00000000i
 Iter:  3 ovlp:   1.00033731 eig:   0.05454213   0.00000000i
 Iter:  4 ovlp:   0.99983488 eig:   0.06068242   0.00000000i
 Iter:  5 ovlp:   1.00003829 eig:   0.05905716   0.00000000i
 Iter:  6 ovlp:   0.99998942 eig:   0.05948990   0.00000000i
 Iter:  7 ovlp:   1.00000278 eig:   0.05937486   0.00000000i
 Iter:  8 ovlp:   0.99999926 eig:   0.05940546   0.00000000i
 Iter:  9 ovlp:   1.00000020 eig:   0.05939732   0.00000000i
 Iter: 10 ovlp:   0.99999995 eig:   0.05939948   0.00000000i
 Iter: 11 ovlp:   1.00000001 eig:   0.05939891   0.00000000i
 Iter: 12 ovlp:   1.00000000 eig:   0.05939906   0.00000000i
 Iter: 13 ovlp:   1.00000000 eig:   0.05939902   0.00000000i
 Iter: 14 ovlp:   1.00000000 eig:   0.05939903   0.00000000i
 Iter: 15 ovlp:   1.00000000 eig:   0.05939903   0.00000000i
 Iter: 16 ovlp:   1.00000000 eig:   0.05939903   0.00000000i
 Iter: 17 ovlp:   1.0000